In [8]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
import random

In [9]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [15]:
tiny_stories = load_dataset("roneneldan/TinyStories", split="train")

print(f"Total size: {len(tiny_stories)}")
print(f"Sample story: {tiny_stories[10]["text"]}")

model_name = "roneneldan/TinyStories-33M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True)
model.eval()

print(model.config)

Total size: 2119719
Sample story: Once upon a time, there was a big car named Dependable. He had a very important job. Dependable would take a family to the park every day. The family had a mom, dad, and a little girl named Lily. They all had a lot of love for each other.

One day, when they got to the park, they saw a big sign that said, "Fun Race Today!" The family was very excited. They knew that Dependable was very fast and could win the race. So, they decided to join the race.

The race started, and Dependable went very fast. The other cars tried to catch up, but Dependable was too quick. In the end, Dependable won the race! The family was so happy and proud of their car. They knew that their love for each other and their trust in Dependable made them win the race. And from that day on, they had even more fun at the park, knowing that they had the fastest and most dependable car around.


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPTNeoConfig {
  "activation_function": "gelu_new",
  "architectures": [
    "GPTNeoForCausalLM"
  ],
  "attention_dropout": 0,
  "attention_layers": [
    "global",
    "local",
    "global",
    "local"
  ],
  "attention_types": [
    [
      [
        "global",
        "local"
      ],
      2
    ]
  ],
  "bos_token_id": 50256,
  "classifier_dropout": 0.1,
  "dtype": "float32",
  "embed_dropout": 0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": null,
  "layer_norm_epsilon": 1e-05,
  "max_position_embeddings": 2048,
  "model_type": "gpt_neo",
  "num_heads": 16,
  "num_layers": 4,
  "output_hidden_states": true,
  "pad_token_id": null,
  "resid_dropout": 0,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.7.0",
  "use_cache": true,
  

In [21]:
N_STORIES = 500

sampled_stories = tiny_stories.shuffle(seed=SEED).select(range(N_STORIES))
print(f"Sampled {len(sampled_stories)} stories")
print(f"Example: {sampled_stories[0]['text'][:200]}\n")

lengths = [len(tokenizer(s["text"])["input_ids"]) for s in sampled_stories]
print(f"Mean tokens per story: {np.mean(lengths):.0f}")
print(f"Min: {min(lengths)}, Max: {max(lengths)}")
print(f"Total tokens: {sum(lengths)}")
print(f"Estimated noun/verb tokens: {sum(lengths) // 2}")

Sampled 500 stories
Example: Tim and Mia like to play in the park. They see a big club on the ground. It is brown and long and heavy.

"Look, a club!" Tim says. "I can lift it!"

He tries to lift the club, but it is too tough. He

Mean tokens per story: 225
Min: 89, Max: 1022
Total tokens: 112638
Estimated noun/verb tokens: 56319


In [22]:
nlp = spacy.load("en_core_web_sm")

def get_pos_tags(text):
    doc = nlp(text)
    # return list of (word, POS tag) pairs
    return [(token.text, token.pos_) for token in doc]

# test on one story
example_tags = get_pos_tags(sampled_stories[0]["text"])
print(example_tags[:20])

[('Tim', 'PROPN'), ('and', 'CCONJ'), ('Mia', 'PROPN'), ('like', 'VERB'), ('to', 'PART'), ('play', 'VERB'), ('in', 'ADP'), ('the', 'DET'), ('park', 'NOUN'), ('.', 'PUNCT'), ('They', 'PRON'), ('see', 'VERB'), ('a', 'DET'), ('big', 'ADJ'), ('club', 'NOUN'), ('on', 'ADP'), ('the', 'DET'), ('ground', 'NOUN'), ('.', 'PUNCT'), ('It', 'PRON')]


In [23]:
pos_tagged_stories = []

for story in sampled_stories:
    doc = nlp(story["text"])
    tokens_and_tags = [(token.text, token.pos_) for token in doc]
    pos_tagged_stories.append(tokens_and_tags)

noun_count = sum(1 for story in pos_tagged_stories 
                 for _, tag in story if tag == "NOUN")
verb_count = sum(1 for story in pos_tagged_stories 
                 for _, tag in story if tag == "VERB")

print(f"Total NOUN tokens: {noun_count}")
print(f"Total VERB tokens: {verb_count}")
print(f"Ratio: {noun_count/verb_count:.2f}")

Total NOUN tokens: 13714
Total VERB tokens: 16479
Ratio: 0.83


In [ ]:
import random

all_tagged_tokens = [
    (story_idx, token_idx, word, tag)
    for story_idx, story in enumerate(pos_tagged_stories)
    for token_idx, (word, tag) in enumerate(story)
    if tag in ("NOUN", "VERB")
]

random.shuffle(all_tagged_tokens)

split = int(0.8 * len(all_tagged_tokens))
train_pool = all_tagged_tokens[:split]
test_set = all_tagged_tokens[split:]

print(f"Train pool: {len(train_pool)} tokens")
print(f"Test set: {len(test_set)} tokens")